## Função: unzip_files
Esta função automatiza o processo de copiar um arquivo compactado do DBFS para um diretório temporário local, realiza a descompactação e retorna o caminho onde os arquivos foram extraídos.

Estrutura e Fluxo Detalhamento dos Argumentos e Retorno Parâmetro: file (Objeto): Espera um objeto que possua os atributos .name (nome do arquivo com extensão) e .path (caminho completo no DBFS, ex: dbfs:/mnt/...).

Retorno: str: O caminho do diretório local onde os arquivos foram extraídos (/dbfs/tmp/extract_...).

## O que cada etapa faz: Definição de Caminhos:

Cria um caminho temporário para o arquivo .zip.
Cria uma pasta de destino exclusiva baseada no nome do arquivo original (removendo a extensão .zip).
Cópia de Arquivo: Utiliza dbutils.fs.cp para mover o arquivo do storage para o "local" do cluster. O prefixo file: é usado para indicar ao sistema que o destino é o sistema de arquivos local.
Extração: Usa o contexto with zipfile.ZipFile para abrir o arquivo em modo de leitura e extrair todo o conteúdo para a pasta de destino.
Tratamento de Erros: Um bloco try-except genérico captura falhas (como arquivo corrompido ou falta de permissão) e imprime o erro no console.

In [0]:
import zipfile
import json
import os
import pytz
from pyspark.sql.types import StructType
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
from datetime import datetime

In [0]:
schema_path = "abfss://config@stgbbb.dfs.core.windows.net/schema"

In [0]:

def unzip_files(file):
    try:
        zip_name = file.name
        local_zip_path = f"/dbfs/tmp/{zip_name}"
        local_extract_path = f"/dbfs/tmp/extract_{zip_name.replace('.zip', '')}"
        
        dbutils.fs.cp(file.path, "file:" + local_zip_path)
        
        with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
            zip_ref.extractall(local_extract_path)
            
    except Exception as e:
        print(e)
    
    return local_extract_path
 

In [0]:

def schema_existe(nome_tabela):
    try:
        dbutils.fs.ls(f"{schema_path}/{nome_tabela}.json")
        return True
    except AnalysisException:
        return False

In [0]:
def save_schema(df, nome_tabela):
    schema_json = df.schema.json()
    caminho = f"{schema_path}/{nome_tabela}.json"
    dbutils.fs.put(caminho, schema_json, overwrite=True)

In [0]:
def enforce_schema(df, nome_tabela):

    if not schema_existe(nome_tabela):
        save_schema(df, nome_tabela)
        return df

    """Lê o JSON do Lake e força o DF a seguir os tipos definidos."""
    caminho = f"{schema_path}/{nome_tabela}.json"
    
    # 1. Ler o arquivo JSON do Storage
    json_str = dbutils.fs.head(caminho)
    stored_schema = StructType.fromJson(json.loads(json_str))
    
    stored_cols_map = {f.name: f.dataType for f in stored_schema}
    
    select_expr = []
    has_schema_changed = False
    
    for col_name in df.columns:
        
        if col_name in stored_cols_map:
            # A coluna JÁ EXISTIA: Forçamos o tipo antigo (Enforcement) para segurança
            target_type = stored_cols_map[col_name]
            # Ex: Se era Date e veio String, tenta converter. Se falhar, vira Null (Safe Cast)
            select_expr.append(F.col(col_name).cast(target_type).alias(col_name))
        else:
            # A coluna é NOVA: Deixamos passar como veio (Evolution)
            select_expr.append(F.col(col_name))
            has_schema_changed = True
    
    df_final = df.select(select_expr)
    
    if has_schema_changed:
        save_schema(df_final, nome_tabela)
        
    return df_final


#FUNÇÃO:clean_and_fill
Esta função é um utilitário de Data Quality para PySpark. Ela resolve o problema de colunas "sujas" que contêm espaços em branco ou strings vazias que não são reconhecidas como nulas nativamente, padronizando-as com um texto amigável.📋 Descrição GeralA função percorre uma lista de colunas selecionadas e realiza um tratamento em duas etapas:Normalização: Remove espaços em branco nas extremidades e converte strings vazias em NULL.Imputação: Substitui todos os valores NULL encontrados por um valor padrão (default: "Não Informado").🛠️ ParâmetrosParâmetroTipoDescriçãodfpyspark.sql.DataFrameO DataFrame original que contém os dados.columnslistUma lista de strings com os nomes das colunas a serem tratadas.replacementstr(Opcional) O texto que deve substituir os valores nulos. Padrão: "Não Informado".🔄 Fluxo de FuncionamentoLoop Iterativo: A função percorre cada nome de coluna fornecido na lista columns.Lógica trim + when:trim(col(column)): Remove espaços (ex: "  " vira "").when(... == "", None): Se o resultado for uma string vazia, injeta um NULL. Isso é crucial porque o Spark não trata "" como nulo automaticamente.Lógica fillna: Após garantir que o que era vazio agora é oficialmente NULL, a função usa o subset para preencher apenas as colunas da lista com o valor de replacement.Feedback: Um print no console confirma quais colunas foram processadas com sucesso.💡 Exemplo de UsoPython# 1. Defina as colunas em uma lista
colunas_vazias = ["NO_BLOCO", "CO_PAIS"]

# 2. Chame a função passando o DataFrame
df_final = clean_and_fill(df, columns=colunas_vazias, replacement="Dado Indisponível")
⚠️ Observações ImportantesTipo de Dado: Esta função foi desenhada para colunas do tipo String. Se aplicada em colunas numéricas (como int ou double), o trim pode falhar ou gerar comportamentos inesperados.Imutabilidade: Como o Spark trabalha com objetos imutáveis, a função redefine a variável df a cada iteração do loop, garantindo que todas as transformações sejam acumuladas.

In [0]:

def clean_and_fill(df, columns, replacement="Não Informado"):
    from pyspark.sql.functions import when, col, trim
    """
    Transforma strings vazias em nulos e preenche nulos com um valor padrão.
    
    Args:
        df: DataFrame do Spark.
        columns: Lista de colunas para tratar (ex: ["regiao", "status"]).
        replacement: Texto que substituirá o nulo.
    """
    for column in columns:
        # Primeiro: trata espaços e transforma "" em None (nulo)
        df = df.withColumn(
            column, 
            when(trim(col(column)) == "", None).otherwise(col(column))
        )
    
    # Segundo: preenche todos os nulos das colunas selecionadas com o texto desejado
    df = df.fillna(replacement, subset=columns)   

    return df

In [0]:
# ==============================
# Config Azure Storage
# ==============================

STORAGE = "stgbbb"
CONTAINER_BRONZE = "bronze"
CONTAINER_SILVER = "silver"

storage_key = dbutils.secrets.get(scope="bbb", key="secret-stg-bbb")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE}.dfs.core.windows.net",
    storage_key
)

In [0]:
# ==============================
# Data automática (Brasil)
# ==============================

fuso_br = pytz.timezone("America/Sao_Paulo")
agora = datetime.now(fuso_br)

# Partição automática (D-2)
particao_hoje = f"ano={agora.year}/mes={agora.month:02d}/dia={agora.day - 2:02d}"

print("Partição usada:", particao_hoje)

In [0]:
# ==============================
# Funções reutilizáveis
# ==============================

def get_bronze_path(dataset):
    return (
        f"abfss://{CONTAINER_BRONZE}@{STORAGE}.dfs.core.windows.net/"
        f"balancacomercial/{particao_hoje}/{dataset}/"
    )

def get_silver_path(dataset):
    return (
        f"abfss://{CONTAINER_SILVER}@{STORAGE}.dfs.core.windows.net/"
        f"balancacomercial/{dataset}/"
    )

def read_bronze(dataset):
    path = get_bronze_path(dataset)
    print(f"Lendo Bronze: {path}")
    return spark.read.parquet(path)

def write_silver(df, dataset):
    path = get_silver_path(dataset)
    print(f"Salvando Silver: {path}")

    df.write.mode("overwrite").parquet(path)